<a href="https://colab.research.google.com/github/KanitAon/ebay-soccer-card-market-analysis/blob/main/src/data_collection/eBay_Market_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# eBay Market Data

## Summary

This notebook collects Premier League football card listings from eBay using the Browse API.

The search focuses on two card brands:

* **Topps**
* **Panini**

The result may include:

* Player
* Club
* Season
* Brand
* Product line
* Card number
* Serial number
* Rookie
* Autograph
* Grade
* Price
* Seller
* Listing URL

The final dataset is exported as **Excel** files.

> **Note:** The result may contain only **Topps and Panini** cards because these are the brands used in the search keywords.

> The price is the current **listing price**, not a confirmed historical sold price.

## Flow

1. **Import libraries**
   Load the Python libraries used in the notebook.

2. **Enter eBay API credentials**
   Enter `client_id` and `client_secret`.

3. **Get access token**
   Use the credentials to request an OAuth access token from eBay.

4. **Set search keywords**
   Create search terms for Premier League cards by season and brand.

   Example:

   ```text
   premier league panini 2025
   premier league topps 2025
   ```

5. **Get eBay listings**
   Request football card listings from the eBay Browse API.

6. **Process the data**
   Convert the API result into a pandas DataFrame.

7. **Extract card information**
   Read the listing title and try to identify player, club, season, product line, rookie, autograph, grade, and other card details.

8. **Check the result**
   Review the number of listings, prices, brands, players, and missing values.

9. **Export the dataset**
   Save the final result as:

   ```text
   ebay_premier_league_cards.xlsx
   ```

### Final Result

The final dataset can be used for Premier League football card market analysis, such as comparing **Topps vs Panini**, player prices, rookie cards, autograph cards, graded cards, and product lines.


## How to use this notebook

In Google Colab, click **Runtime → Run all**.

Then:

1. Enter your `client_id`
2. Enter your `client_secret`
3. Let the notebook get the eBay data
4. The notebook will create the output files
5. The Excel file will download automatically


## 1. Import libraries

This cell imports the libraries used in this notebook.

In [ ]:
import requests
import base64
import time
import re
import math
import pandas as pd

from datetime import datetime, timezone
from getpass import getpass

pd.set_option("display.max_columns", None)

print("Libraries loaded ✅")

## 2. Enter your eBay API credentials

To use the eBay API, you need an **eBay Developer account** and API credentials.

1. Go to the [eBay Developers Program](https://developer.ebay.com/)
2. Sign in or create a developer account
3. Open **Application Keysets**
4. Create or use a **Production** keyset
5. Copy:
   - **App ID** → `client_id`
   - **Cert ID** → `client_secret`
6. Enter both values in the form below

In [ ]:

client_id = "" #@param {type:"string"}
client_secret = "" #@param {type:"string"}

if not client_id or not client_secret:
    raise ValueError("Please enter both client_id and client_secret")

print("Credentials entered successfully ✅")

## 3. Get an eBay access token

eBay uses your `client_id` and `client_secret` to create an access token.

The token is saved in the variable `access_token` and is used to call the Browse API.


In [ ]:
token_url = "https://api.ebay.com/identity/v1/oauth2/token"

basic_auth = base64.b64encode(
    f"{client_id}:{client_secret}".encode()
).decode()

response = requests.post(
    token_url,
    headers={
        "Content-Type": "application/x-www-form-urlencoded",
        "Authorization": f"Basic {basic_auth}",
    },
    data={
        "grant_type": "client_credentials",
        "scope": "https://api.ebay.com/oauth/api_scope",
    },
    timeout=30,
)

response.raise_for_status()
access_token = response.json()["access_token"]

print("eBay access token created ✅")

## 4. Set the search

This is the main section you may want to change.

- `search_seasons` = seasons or years you want to search
- `max_listings_per_brand` = maximum number of listings for each brand
- This notebook searches for **Panini** and **Topps**


In [ ]:
search_seasons = [
    "2023",
    "2023-24",
    "2024-25",
    "2025",
    "2025-26",
    "2026",
]

max_listings_per_brand = 5000

panini_search_queries = [f"premier league panini {season}" for season in search_seasons]
topps_search_queries = [f"premier league topps {season}" for season in search_seasons]

print("Panini queries:", panini_search_queries)
print("Topps queries :", topps_search_queries)

## 5. Function to get data from eBay

The `collect_ebay_items()` function:

- Sends a request to the eBay Browse API
- Gets 50 listings at a time

If the API returns an error, the code shows the query and offset to make debugging easier.


In [ ]:
def collect_ebay_items(query_list, token, cap=2000):
    url = "https://api.ebay.com/buy/browse/v1/item_summary/search"

    seen = set()
    items = []

    headers = {
        "Authorization": f"Bearer {token}",
        "X-EBAY-C-MARKETPLACE-ID": "EBAY_US",
        "Accept": "application/json",
    }

    for query in query_list:

        if len(items) >= cap:
            break

        print(f"Searching eBay for: {query}")
        offset = 0

        while len(items) < cap:

            params = {
                "q": query,
                "limit": 50,
                "offset": offset,
                "sort": "-price",
            }

            response = requests.get(
                url,
                headers=headers,
                params=params,
                timeout=30,
            )

            if not response.ok:
                print(
                    f"Request failed | query={query} | "
                    f"offset={offset} | status={response.status_code}"
                )
                print(response.text[:500])
                response.raise_for_status()

            page = response.json().get("itemSummaries", [])

            if not page:
                break

            for item in page:
                key = (
                    item.get("legacyItemId")
                    or item.get("itemId")
                    or item.get("title")
                )

                if key not in seen:
                    seen.add(key)
                    items.append(item)

                    if len(items) >= cap:
                        break

            offset += 50
            time.sleep(0.25)

    return items[:cap]

## 6. Get data

This cell starts calling the eBay API.

In [ ]:
panini_results = collect_ebay_items(
    panini_search_queries,
    token=access_token,
    cap=max_listings_per_brand,
)

topps_results = collect_ebay_items(
    topps_search_queries,
    token=access_token,
    cap=max_listings_per_brand,
)

print()
print(f"Panini listings : {len(panini_results):,}")
print(f"Topps listings  : {len(topps_results):,}")
print(f"Total listings  : {len(panini_results) + len(topps_results):,}")

## 7. Functions to read card details from the listing title

This section uses regular expressions and keywords to extract information such as:

- Player
- Club
- Season / Year
- Product line
- Card number
- Numbered card
- Grade
- Rookie
- Autograph
- Patch / Relic

> The extracted values come from the listing title, so some results may not be perfect. Check important records before using them as final data.


In [ ]:
import pandas as pd
import re
from datetime import datetime, timezone

KNOWN_PRODUCTS = [
    "Prizm", "Donruss", "Select", "Optic", "Mosaic", "Score", "Contenders",
    "Rookies & Stars", "Rookies and Stars", "Absolute", "Chronicles", "Elite",
    "Playoff", "Immaculate", "National Treasures", "Flawless", "Origins",
    "Obsidian", "Spectra", "Legacy", "Momentum", "Instant", "Luminance",
    "Prestige", "Certified", "Leaf", "Phoenix", "Black",
]

STOP = set("""panini topps chrome prizm donruss select optic mosaic score contenders
rookies stars absolute chronicles elite playoff immaculate national treasures
flawless origins obsidian spectra legacy momentum instant luminance prestige
certified leaf phoenix black red green gold silver blue orange yellow white
pink purple wave disco turbocharged shock shockprizm hologold case box pack
lot lots cards card auto autograph autographed signed signature swatch patch
memorabilia gem mint near nm condition graded ungraded psa bgs sgc cgc csg
beckett hga slab pop micr roc sealed unopened factory serial numbered low
titanium rc rookie rpa rps postseason ticket base parallel insert refractor
true on in the of a an with for and to or world nfl football soccer sport team
jersey jerseypatch rookiecard collection master complete super spc con ssp
hothtohtoh player day exclusive veteran veterans pro bowl hof hall fame mvp
all pro win triple threat true gold silver_flash rpa rookie premier league
match attax adrenalyn stickers mystery bag guaranteed 1 2 3 2016 2017 2018
2019 2020 2021 2022 2023 2024 2025 2000 2001 2002 2003 2004 2005 2006 2007
2008 2009 2010 2011 2012 2013 2014 2015 1990 1991 1992 1993 1994 1995 1996
1997 1998 1999 cardinals falcons panthers bears cowboys lions packers rams
chargers vikings saints giants jets eagles seahawks buccaneers commanders
ravens bills bengals browns broncos texans colts jaguars chiefs raiders
dolphins patriots titans redskins oilers steelers arizona atlanta carolina
chicago dallas detroit green los angeles minnesota new orleans york
philadelphia san francisco seattle tampa washington baltimore buffalo
cincinnati cleveland denver houston indianapolis kansas las vegas miami
england oakland san diego tennessee st louis upc code shaped travel super
bowl""".split())

CLUBS = [
    ("Arsenal", r"arsenal"), ("Man City", r"manchester city|man city"),
    ("Man United", r"manchester united|man utd"), ("Liverpool", r"liverpool"),
    ("Chelsea", r"chelsea"), ("Spurs", r"tottenham|spurs"), ("Newcastle", r"newcastle"),
    ("Aston Villa", r"aston villa"), ("West Ham", r"west ham"), ("Brighton", r"brighton"),
    ("Crystal Palace", r"crystal palace"), ("Everton", r"everton"), ("Fulham", r"fulham"),
    ("Brentford", r"brentford"), ("Bournemouth", r"bournemouth"), ("Wolves", r"wolverhampton|wolves"),
    ("Leicester", r"leicester"), ("Leeds", r"leeds"), ("Nottingham Forest", r"nottingham forest"),
    ("Southampton", r"southampton"), ("Burnley", r"burnley"), ("Sheffield Utd", r"sheffield united|sheffield utd"),
    ("Sunderland", r"sunderland"), ("Norwich", r"norwich"), ("Watford", r"watford"),
    ("West Brom", r"west brom"), ("Stoke", r"stoke"), ("Hull", r"hull city|hull"),
    ("Ipswich", r"ipswich"), ("QPR", r"queens park|qpr"), ("Derby", r"derby county|derby"),
    ("Cardiff", r"cardiff"), ("Swansea", r"swansea"),
]

POSITIONS = [
    "GK", "DF", "MF", "FW", "CF", "ST", "CM", "CDM", "CAM", "LW", "RW",
    "LM", "RM", "WB", "SB", "CB", "LB", "RB", "QB", "TE", "DE", "DT",
    "NT", "ILB", "OLB", "MLB", "FS", "SS", "K", "P", "EDGE",
]

POSITION_RE = re.compile(r"\b(" + "|".join(POSITIONS) + r")\b")

def parse_year(title):
    m = re.search(r"\b(19|20)\d{2}\b", title)
    return int(m.group(0)) if m else ""

def parse_season(title):
    m = re.search(r"\b(20\d{2})[- ](\d{2})\b", title)
    return f"{m.group(1)}-{m.group(2)}" if m else ""

def parse_product(title):
    for p in KNOWN_PRODUCTS:
        if re.search(r"\b" + re.escape(p) + r"\b", title, re.I):
            return p
    return ""

def parse_grade(title):
    m = re.search(r"\b(?:PSA|BGS|SGC|CGC)\s*-?\s*(\d{2})(?:\D|$)", title, re.I)
    return int(m.group(1)) if m else ""

def parse_club(title):
    for club, pat in CLUBS:
        if re.search(pat, title, re.I):
            return club
    return ""

def parse_card_number(title):
    m = re.search(r"#\s*(\d{1,4})", title)
    return int(m.group(1)) if m else ""

def parse_serial(title):
    m = re.search(r"(\d{1,4})[/(](\d{1,4})", title)
    return int(m.group(2)) if m else ""

def has_emoji(text):
    return bool(re.search(r"[\U0001F300-\U0001FAFF\u2600-\u27BF]", text))

def parse_position(title):
    if not title:
        return ""
    m = POSITION_RE.search(title)
    return m.group(1).upper() if m else ""

def parse_player(title):
    text = re.sub(r"[\U0001F300-\U0001FAFF\u2600-\u27BF#]", " ", title)
    text = re.sub(r"[^A-Za-z. ]", " ", text)
    words = re.findall(r"[A-Za-z][A-Za-z.'-]*", text)
    runs, run = [], []
    for w in words:
        key = w.translate(str.maketrans("", "", ".'-")).lower()
        if key in STOP or key.isdigit() or len(key) <= 1:
            if run:
                runs.append(run)
                run = []
            continue
        run.append(w)
    if run:
        runs.append(run)
    if not runs:
        return ""
    good = [r for r in runs if any(w[0].isupper() or w.isupper() for w in r)]
    if not good:
        good = runs
    multi = [r for r in good if len(r) >= 2]
    best = max(multi or good, key=len)
    if len(best) == 1 and any(len(r) >= 2 for r in good):
        return ""
    return " ".join(best)

def parse_end(end_date):
    if not end_date:
        return "", ""
    try:
        dt = datetime.fromisoformat(end_date.replace("Z", "+00:00").rstrip("Z"))
        return (dt - datetime.now(timezone.utc)).days, dt.weekday()
    except ValueError:
        return "", ""

def items_to_dataframe(items):
    rows = []
    for it in items:
        price = float(it.get("price", {}).get("value") or 0)
        title = it["title"]
        seller = it.get("seller", {}) or {}
        cats = it.get("categories", []) or []
        buying = it.get("buyingOptions", []) or []
        days_left, end_wd = parse_end(it.get("itemEndDate", ""))
        rows.append({
            "title": title,
            "player": parse_player(title),
            "position": parse_position(title),
            "club": parse_club(title),
            "season": parse_season(title),
            "brand": "",
            "product_line": parse_product(title),
            "year": parse_year(title),
            "card_number": parse_card_number(title),
            "serial": parse_serial(title),
            "price": price,
            "price_log": float(__import__("math").log1p(price)),
            "currency": it.get("price", {}).get("currency", ""),
            "condition": it.get("condition", ""),
            "graded": bool(re.search(r"\b(PSA|BGS|SGC|CGC|CSG|Beckett|HGA)\b|graded", title, re.I)),
            "grade": parse_grade(title),
            "autograph": bool(re.search(r"auto|autograph|signed|signature|swatch|patch", title, re.I)),
            "patch": bool(re.search(r"patch|jsy|jersey|swatch|rpa|relic", title, re.I)),
            "rookie": bool(re.search(r"rookie|\bRC\b|rpa", title, re.I)),
            "buying_options": ",".join(buying),
            "is_auction": "AUCTION" in buying,
            "bids": it.get("bidCount", 0),
            "end_date": it.get("itemEndDate", ""),
            "days_until_end": days_left,
            "end_weekday": end_wd,
            "num_images": len(it.get("thumbnailImages", [])),
            "category": cats[0].get("categoryName", "") if cats else "",
            "country": (it.get("itemLocation", {}) or {}).get("country", ""),
            "seller": seller.get("username", ""),
            "seller_feedback_count": seller.get("feedbackScore", 0),
            "seller_feedback_pct": seller.get("feedbackPercent", ""),
            "title_length": len(title),
            "has_emoji": has_emoji(title),
            "url": it.get("itemWebUrl", ""),
        })
    if not rows:
        return pd.DataFrame(columns=['title', 'player', 'position', 'club', 'season', 'brand', 'product_line', 'year', 'card_number', 'serial', 'price', 'price_log', 'currency', 'condition', 'graded', 'grade', 'autograph', 'patch', 'rookie', 'buying_options', 'is_auction', 'bids', 'end_date', 'days_until_end', 'end_weekday', 'num_images', 'category', 'country', 'seller', 'seller_feedback_count', 'seller_feedback_pct', 'title_length', 'has_emoji', 'url'])

    df = pd.DataFrame(rows)
    df = df.sort_values("price", ascending=False).reset_index(drop=True)

    # Keep only listings with a price higher than $10
    return df[df["price"] > 10].reset_index(drop=True)

## 8. Create DataFrames

This section converts the eBay JSON data into `pandas DataFrame` tables.

You will get 3 main tables:

- `panini_df`
- `topps_df`
- `all_cards_df` = Panini and Topps combined


In [ ]:
panini_df = items_to_dataframe(panini_results)
panini_df["brand"] = "panini"

topps_df = items_to_dataframe(topps_results)
topps_df["brand"] = "topps"

all_cards_df = pd.concat(
    [panini_df, topps_df],
    ignore_index=True,
)

for brand_name, data in [
    ("Panini", panini_df),
    ("Topps", topps_df),
]:
    if len(data) == 0:
        print(f"{brand_name}: 0 rows")
    else:
        print(
            f"{brand_name}: {len(data):,} rows | "
            f"median ${data['price'].median():,.2f} | "
            f"max ${data['price'].max():,.2f}"
        )

print(f"Total: {len(all_cards_df):,} rows")
all_cards_df.head(10)

## 9. Quick data check

This section gives a simple overview of the data:

- Number of rows and columns
- Missing values
- Price statistics
- Number of listings by brand
- Most common product lines
- Most common extracted players


In [ ]:
print("Shape:", all_cards_df.shape)

if all_cards_df.empty:
    print("No data returned.")
else:
    print("\nMissing values:")
    missing = all_cards_df.isna().sum()
    print(missing[missing > 0])

    print("\nPrice stats:")
    print(all_cards_df["price"].describe().round(2))

    print("\nRows by brand:")
    print(all_cards_df["brand"].value_counts())

    print("\nTop product lines:")
    print(all_cards_df["product_line"].value_counts().head(10))

    print("\nTop parsed players:")
    print(all_cards_df["player"].value_counts().head(10))

    print("\nCard flags:")
    print(
        all_cards_df[["patch", "autograph", "rookie", "graded"]]
        .astype(int)
        .sum()
    )

## 10. Export to Excel

This cell saves files:

- `ebay_premier_league_cards.xlsx`

When you run it in Google Colab, the Excel file will download automatically.


In [ ]:
OUTPUT_XLSX = "ebay_premier_league_cards.xlsx"

all_cards_df.to_excel(
    OUTPUT_XLSX,
    index=False,
)

print(f"Saved: {OUTPUT_XLSX} ✅")

# Download the Excel file automatically when using Google Colab
try:
    from google.colab import files
    files.download(OUTPUT_XLSX)
except ImportError:
    print("Not running in Google Colab. Files were saved locally.")